In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 16.0,
    'font.family': 'serif',
    'font.serif': 'Palatino',
    'axes.titlesize': 'medium',
    'figure.titlesize': 'large',
    'legend.fontsize': 'medium',
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'figure.autolayout': True,
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{siunitx}[=v2]",
})

from pathlib import Path

PLOT_ROOT = Path.cwd() / "plots"
PLOT_ROOT.mkdir(exist_ok=True)

In [ ]:
import sys
import numpy as np
import pandas as pd
import geopandas as gpd
import scipy.io as sio

sys.path.insert(0, str(Path.cwd()))
from arsw_python.recover_fundamentals import run_calcal_TD, save_results

# ── Repository tree ───────────────────────────────────────────────────────────
REPO_ROOT    = Path.cwd().parent
ARSW_TOOLKIT = REPO_ROOT / "ARSW2015" / "ARSW2015-toolkit"

# ── Data inputs ───────────────────────────────────────────────────────────────
MAT_PATH_TD    = ARSW_TOOLKIT / "matlab" / "data" / "input" / "prepdata_big_TD.mat"
TTM_CLEAN_PATH = Path.cwd() / "TTM" / "tt06_user_preprocessed.parquet"

# ── Saved results ─────────────────────────────────────────────────────────────
USER_NPZ = ARSW_TOOLKIT / "matlab" / "data" / "output" / "calcal_1c_results.npz"
ORIG_NPZ = ARSW_TOOLKIT / "matlab" / "data" / "output" / "calcal_1d_orig_results.npz"

# ── Shapefiles ────────────────────────────────────────────────────────────────
BLOCKS_SHP_FULL = ARSW_TOOLKIT / "shapefile" / "Berlin4matlab.shp"
STREETS_SHP     = REPO_ROOT / "Data" / "Shapefiles-2022" / "Berlin" / "TransportNetworkParts2006" / "Streets.shp"

# ── Base map shapefiles (ARSW toolkit) ──────────────────────────────────────
BEZIRKE_SHP = ARSW_TOOLKIT / "shapefile" / "Bezirke23.shp"
WATER_SHP   = ARSW_TOOLKIT / "shapefile" / "BerlinWater.shp"
GREEN_SHP   = ARSW_TOOLKIT / "shapefile" / "BerlinGreen.shp"

# ── Parameters — must match task_1c exactly ───────────────────────────────────
EPSILON_HAT = 6.83
KAPPAEPS    = 0.07
ALPHA       = 0.80
BETA        = 0.75

print(f"Parameters: ε = {EPSILON_HAT},  κε = {KAPPAEPS},  κ = {KAPPAEPS/EPSILON_HAT:.6f}")
print(f"Streets.shp present: {STREETS_SHP.exists()}")

## Task 1(d) — Comparing Recovered Fundamentals: Original ARSW TTM vs User TTM

**Objective.** The structural fundamentals $\tilde{A}_j$ (adjusted productivity) and
$\tilde{B}_i$ (adjusted amenity) are inverted from the data conditional on the travel
time matrix. Differences in $\tau_{ij}$ between the original ARSW matrix and the
user-computed multimodal TTM propagate into the recovered fundamentals through two
channels:

- **Productivity** $\tilde{A}_j$: identified from the wage fixed-point system
  (ARSW eq. S.44). Any systematic difference in commuting access to workplace $j$
  shifts the implied productivity residual.
- **Amenity** $\tilde{B}_i$: inverted from ARSW eq. S.47 via commuting market access
  $\mathrm{CMA}_i = \sum_j e^{-\varepsilon\kappa\tau_{ij}} w_j^\varepsilon$.
  A higher CMA (shorter travel times to better-paying workplaces) reduces the amenity
  residual needed to rationalize observed residential sorting.

The comparison maps show where the user TTM leads to systematically higher or lower
fundamentals, revealing how TTM construction choices affect structural inference.

In [4]:
# ── Validate and load user results (from task_1c) ────────────────────────────
if not USER_NPZ.exists():
    raise FileNotFoundError(
        f"User TTM results not found: {USER_NPZ}\n"
        "Run task_1c.ipynb first to generate and save the calibration output."
    )

d_user = np.load(str(USER_NPZ))
NOBS06 = int(d_user["nobs06"])

A_user = d_user["A06"].copy()
B_user = d_user["B06"].copy()

with np.errstate(divide='ignore', invalid='ignore'):
    logA_user = np.where(A_user > 0, np.log(A_user), np.nan)
    logB_user = np.where(B_user > 0, np.log(B_user), np.nan)

print(f"User TTM results loaded: nobs06 = {NOBS06}")
print(f"  A06: {(A_user > 0).sum()} positive blocks,  "
      f"geomean = {np.exp(np.nanmean(logA_user)):.4f}")
print(f"  B06: {(B_user > 0).sum()} positive blocks")

User TTM results loaded: nobs06 = 12309
  A06: 9437 positive blocks,  geomean = 1.0000
  B06: 11863 positive blocks


In [5]:
# ── Cache-aware: run original calibration only if not already saved ───────────
if ORIG_NPZ.exists():
    print(f"Loading cached original ARSW results: {ORIG_NPZ.name}")
    d_orig = np.load(str(ORIG_NPZ))
else:
    if not MAT_PATH_TD.exists():
        raise FileNotFoundError(
            f"prepdata_big_TD.mat not found: {MAT_PATH_TD}\n"
            "Download from: https://box.hu-berlin.de/f/54d2f718ec8644e5888f/?dl=1\n"
            f"Save to: {MAT_PATH_TD.parent}"
        )
    print("Running calibration with original ARSW tt06 (approx 5-10 min) ...")
    results_orig = run_calcal_TD(
        mat_path_TD=MAT_PATH_TD,
        user_ttm_path=None,          # <- None uses embedded tt06 from prepdata_big_TD.mat
        epsilon=EPSILON_HAT,
        kappaeps=KAPPAEPS,
        alpha=ALPHA,
        beta=BETA,
        verbose=True,
    )
    save_results(results_orig, ORIG_NPZ)
    print(f"Cached to: {ORIG_NPZ}")
    d_orig = np.load(str(ORIG_NPZ))

A_orig = d_orig["A06"].copy()
B_orig = d_orig["B06"].copy()

with np.errstate(divide='ignore', invalid='ignore'):
    logA_orig = np.where(A_orig > 0, np.log(A_orig), np.nan)
    logB_orig = np.where(B_orig > 0, np.log(B_orig), np.nan)

n_orig = int(d_orig["nobs06"])
print(f"\nOriginal ARSW results: nobs06 = {n_orig}")
print(f"  A06: {(A_orig > 0).sum()} positive blocks,  "
      f"geomean = {np.exp(np.nanmean(logA_orig)):.4f}")
print(f"  B06: {(B_orig > 0).sum()} positive blocks")

if n_orig != NOBS06:
    raise ValueError(
        f"nobs06 mismatch: user={NOBS06}, original={n_orig}. "
        "Both calibrations must operate on the same prepdata_big_TD.mat."
    )

Loading cached original ARSW results: calcal_1d_orig_results.npz

Original ARSW results: nobs06 = 12309
  A06: 9437 positive blocks,  geomean = 1.0000
  B06: 11863 positive blocks


In [ ]:
def _clean_shapefile(gdf: gpd.GeoDataFrame, target_crs: int = 25833) -> gpd.GeoDataFrame:
    """
    Six-step geometry cleaning — byte-for-byte identical to TTM/Final.py and task_1c.ipynb.
    Must not deviate: the result arrays are assigned positionally by block index.
    """
    def _valid_coords(geom):
        try:
            b = geom.bounds
            return len(b) == 4 and not (np.any(np.isnan(b)) or np.any(np.isinf(b)))
        except Exception:
            return False
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].reset_index(drop=True)
    gdf = gdf[gdf.geometry.apply(_valid_coords)].reset_index(drop=True)
    gdf["geometry"] = gdf.geometry.make_valid()
    gdf = gdf.to_crs(epsg=target_crs)
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].reset_index(drop=True)
    gdf = gdf[gdf.geometry.apply(_valid_coords)].reset_index(drop=True)
    return gdf


print("Loading Berlin4matlab.shp ...")
gdf_berlin = _clean_shapefile(gpd.read_file(str(BLOCKS_SHP_FULL)))
n_shp = len(gdf_berlin)

if n_shp != NOBS06:
    raise ValueError(
        f"Shapefile has {n_shp} blocks but nobs06 = {NOBS06}. "
        "Geometry cleaning mismatch."
    )
print(f"  Blocks: {n_shp}  (matches nobs06)")

# ── Load base map layers for complete Berlin background ────────────────────────
print("Loading base map layers ...")
gdf_bezirke = gpd.read_file(str(BEZIRKE_SHP)).to_crs(epsg=25833)
gdf_water   = gpd.read_file(str(WATER_SHP)).to_crs(epsg=25833)
gdf_green   = gpd.read_file(str(GREEN_SHP)).to_crs(epsg=25833)
print(f"  Districts: {len(gdf_bezirke)},  Green areas: {len(gdf_green)},  Water: loaded")

# ── Load streets for overlay ───────────────────────────────────────────────────
streets_gdf = None
if STREETS_SHP.exists():
    print("Loading Streets.shp for overlay ...")
    streets_gdf = gpd.read_file(str(STREETS_SHP)).to_crs(epsg=25833)
    print(f"  Street segments: {len(streets_gdf)}")
else:
    print(f"Streets.shp not found — overlay will be skipped")

# ── Assign result vectors by positional index ─────────────────────────────────
gdf_berlin["logA_orig"] = logA_orig
gdf_berlin["logA_user"] = logA_user
gdf_berlin["logB_orig"] = logB_orig
gdf_berlin["logB_user"] = logB_user

# ── Differences: user minus original (NaN where either is non-positive) ───────
both_A = (A_orig > 0) & (A_user > 0)
both_B = (B_orig > 0) & (B_user > 0)

dlogA = np.where(both_A, logA_user - logA_orig, np.nan)
dlogB = np.where(both_B, logB_user - logB_orig, np.nan)

gdf_berlin["dlogA"] = dlogA
gdf_berlin["dlogB"] = dlogB

# ── Diagnostics ───────────────────────────────────────────────────────────────
for name, arr, n_valid in [
    ("Delta log A", dlogA, both_A.sum()),
    ("Delta log B", dlogB, both_B.sum()),
]:
    finite = arr[~np.isnan(arr)]
    print(f"\n{name}: {n_valid} blocks with valid diff")
    print(f"  mean = {finite.mean():.4f},  std = {finite.std():.4f}")
    print(f"  range: [{finite.min():.3f},  {finite.max():.3f}]")

In [ ]:
# ── Shared color scales ───────────────────────────────────────────────────────
# Panels 0-1: shared vmin/vmax computed across BOTH original and user arrays.
vmin_A = float(np.nanmin([logA_orig, logA_user]))
vmax_A = float(np.nanmax([logA_orig, logA_user]))

vmin_B = float(np.nanmin([logB_orig, logB_user]))
vmax_B = float(np.nanmax([logB_orig, logB_user]))

# Panel 2: symmetric diverging scale about 0
_dA_fin = dlogA[~np.isnan(dlogA)]
_dB_fin = dlogB[~np.isnan(dlogB)]
vabs_A = float(np.abs(_dA_fin).max()) if len(_dA_fin) > 0 else 0.1
vabs_B = float(np.abs(_dB_fin).max()) if len(_dB_fin) > 0 else 0.1

MISSING = {"color": "none", "label": "No data"}

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 24))

def _base_layers(ax):
    """Draw complete Berlin base map: districts → green → water."""
    gdf_bezirke.plot(ax=ax, color="#e0e0e0", edgecolor="#aaaaaa", linewidth=0.3, zorder=1)
    gdf_green.plot(ax=ax, color="#c8e6c9", edgecolor="none", alpha=0.7, zorder=2)
    gdf_water.plot(ax=ax, color="#b3cde3", edgecolor="none", zorder=3)

def _streets(ax):
    if streets_gdf is not None:
        streets_gdf.plot(ax=ax, color="white", linewidth=0.15, alpha=0.4, zorder=5)

# ── Row 0: log A ──────────────────────────────────────────────────────────────
for c, (col, title) in enumerate([
    ("logA_orig", r"Original ARSW TTM --- $\log\tilde{A}_j$"),
    ("logA_user", r"User TTM --- $\log\tilde{A}_j$"),
]):
    ax = axes[0, c]
    _base_layers(ax)
    gdf_berlin.plot(
        column=col, ax=ax, cmap="YlOrRd", vmin=vmin_A, vmax=vmax_A,
        legend=True, missing_kwds=MISSING,
        legend_kwds={"label": r"$\log\tilde{A}$", "shrink": 0.70,
                     "orientation": "vertical", "pad": 0.02},
        zorder=4,
    )
    _streets(ax)
    ax.set_title(title, pad=8, fontsize=12)
    ax.set_axis_off()

# ── Row 1: log B ──────────────────────────────────────────────────────────────
for c, (col, title) in enumerate([
    ("logB_orig", r"Original ARSW TTM --- $\log\tilde{B}_i$"),
    ("logB_user", r"User TTM --- $\log\tilde{B}_i$"),
]):
    ax = axes[1, c]
    _base_layers(ax)
    gdf_berlin.plot(
        column=col, ax=ax, cmap="YlOrRd", vmin=vmin_B, vmax=vmax_B,
        legend=True, missing_kwds=MISSING,
        legend_kwds={"label": r"$\log\tilde{B}$", "shrink": 0.70,
                     "orientation": "vertical", "pad": 0.02},
        zorder=4,
    )
    _streets(ax)
    ax.set_title(title, pad=8, fontsize=12)
    ax.set_axis_off()

# ── Row 2: differences ────────────────────────────────────────────────────────
for c, (col, vabs, title, cbar_label) in enumerate([
    ("dlogA", vabs_A,
     r"$\Delta\log\tilde{A}_j = \log\tilde{A}^{\rm user}_j - \log\tilde{A}^{\rm ARSW}_j$",
     r"$\Delta\log\tilde{A}$"),
    ("dlogB", vabs_B,
     r"$\Delta\log\tilde{B}_i = \log\tilde{B}^{\rm user}_i - \log\tilde{B}^{\rm ARSW}_i$",
     r"$\Delta\log\tilde{B}$"),
]):
    ax = axes[2, c]
    _base_layers(ax)
    gdf_berlin.plot(
        column=col, ax=ax, cmap="RdBu_r", vmin=-vabs, vmax=vabs,
        legend=True, missing_kwds=MISSING,
        legend_kwds={"label": cbar_label, "shrink": 0.70,
                     "orientation": "vertical", "pad": 0.02},
        zorder=4,
    )
    _streets(ax)
    ax.set_title(title, pad=8, fontsize=11)
    ax.set_axis_off()

# ── Row labels and suptitle ───────────────────────────────────────────────────
row_labels = [
    r"\textbf{Panel 1:} Adjusted Productivity $\log\tilde{A}_j$",
    r"\textbf{Panel 2:} Adjusted Amenity $\log\tilde{B}_i$",
    r"\textbf{Panel 3:} Difference (User $-$ ARSW)",
]
for r, label in enumerate(row_labels):
    axes[r, 0].set_ylabel(label, fontsize=11, labelpad=8)

plt.suptitle(
    r"Task 1(d): Fundamentals comparison --- Original ARSW TTM vs User-Computed TTM"
    "\n"
    rf"($\hat{{\varepsilon}} = {EPSILON_HAT}$, "
    rf"$\hat{{\kappa}} = {KAPPAEPS/EPSILON_HAT:.4f}$, "
    rf"$\alpha = {ALPHA}$, $\beta = {BETA}$)",
    fontsize=13, y=1.005,
)

plt.tight_layout(rect=[0, 0, 1, 1])

save_path = PLOT_ROOT / "task_1d_fundamentals_comparison.png"
fig.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {save_path}")

In [8]:
print("=" * 72)
print("  Task 1(d) -- Summary: Fundamentals Comparison")
print("=" * 72)
print()
print(f"  Parameters:  epsilon={EPSILON_HAT},  kappa={KAPPAEPS/EPSILON_HAT:.6f},  "
      f"alpha={ALPHA},  beta={BETA}")
print(f"  nobs06 = {NOBS06}  (East + West Berlin, 2006)")
print()
print("  Productivity A (positive blocks):")
print(f"    Original TTM : {(A_orig > 0).sum():5d}  geomean = "
      f"{np.exp(np.nanmean(logA_orig)):.4f}")
print(f"    User TTM     : {(A_user > 0).sum():5d}  geomean = "
      f"{np.exp(np.nanmean(logA_user)):.4f}")
_dA = dlogA[~np.isnan(dlogA)]
print(f"    Delta log A  : mean={_dA.mean():.4f}, std={_dA.std():.4f}, "
      f"p5={np.percentile(_dA, 5):.3f}, p95={np.percentile(_dA, 95):.3f}")
print()
print("  Amenity B (positive blocks):")
print(f"    Original TTM : {(B_orig > 0).sum():5d}")
print(f"    User TTM     : {(B_user > 0).sum():5d}")
_dB = dlogB[~np.isnan(dlogB)]
print(f"    Delta log B  : mean={_dB.mean():.4f}, std={_dB.std():.4f}, "
      f"p5={np.percentile(_dB, 5):.3f}, p95={np.percentile(_dB, 95):.3f}")
print()
print(f"  Original results cached : {ORIG_NPZ.name}")
print(f"  Figure saved            : plots/task_1d_fundamentals_comparison.png")
print()
print("  Interpretation:")
print("    Delta log A > 0 at j  =>  User TTM implies higher adjusted productivity")
print("      at block j. Consistent with user TTM overestimating commuting times TO j.")
print()
print("    Delta log B > 0 at i  =>  User TTM implies higher adjusted amenity at i.")
print("      Follows from CMA_i being lower in user TTM (longer average travel times")
print("      to workplaces): same residential employment with lower CMA requires")
print("      a larger amenity residual to rationalise observed sorting.")
print("=" * 72)

  Task 1(d) -- Summary: Fundamentals Comparison

  Parameters:  epsilon=6.83,  kappa=0.010249,  alpha=0.8,  beta=0.75
  nobs06 = 12309  (East + West Berlin, 2006)

  Productivity A (positive blocks):
    Original TTM :  9437  geomean = 1.0000
    User TTM     :  9437  geomean = 1.0000
    Delta log A  : mean=0.0000, std=0.0303, p5=-0.049, p95=0.049

  Amenity B (positive blocks):
    Original TTM : 11863
    User TTM     : 11863
    Delta log B  : mean=0.0545, std=0.0650, p5=-0.012, p95=0.176

  Original results cached : calcal_1d_orig_results.npz
  Figure saved            : plots/task_1d_fundamentals_comparison.png

  Interpretation:
    Delta log A > 0 at j  =>  User TTM implies higher adjusted productivity
      at block j. Consistent with user TTM overestimating commuting times TO j.

    Delta log B > 0 at i  =>  User TTM implies higher adjusted amenity at i.
      Follows from CMA_i being lower in user TTM (longer average travel times
      to workplaces): same residential employ